# Regime A calibration (tuning)

Turn the `WorkloadConfig` knobs until `generate()` output passes the frozen
`spec/regime_A.json` tolerances. `validate()` returns both pass/fail **and** the
MSM objective **Q** to minimize.

Workflow: edit the **Tweak** cell -> re-run it -> watch Q drop -> freeze when seeds pass.

In [ ]:
import dataclasses
from pathlib import Path
import numpy as np
import workload_gen as wg

# Robust to cwd: locate the frozen spec next to the package, not via a relative path.
SPEC = Path(wg.__file__).parent / "spec" / "regime_A.json"

def evaluate(cfg, seeds=range(5), show=True):
    """Generate + validate over several seeds. Prints the seed-0 table and a
    multi-seed summary (a setting should pass across seeds, not one lucky draw).
    Returns mean Q over the seeds."""
    reports = [wg.validate(wg.generate(cfg, seed=s), SPEC) for s in seeds]
    Qs = [r.distance for r in reports]
    n_pass = sum(r.passed for r in reports)
    if show:
        print(reports[0])
        print(f"
mean Q over {len(Qs)} seeds = {np.mean(Qs):.4g} "
              f"(min {min(Qs):.3g}, max {max(Qs):.3g}) | passing: {n_pass}/{len(Qs)}")
    return float(np.mean(Qs)), reports

## Knob -> stat reference

| If this stat is off | Turn this knob | Direction |
|---|---|---|
| kurtosis >> 25, ramp_abs_mean too low | `noise_amp_W` (+ `_job_profile` in generator.py) | up: within-job dynamics |
| peaks too high / p95 stacking | `arrival_rate_per_s` | down: less overlap |
| marginal too idle (median low) | `arrival_rate_per_s` x `duration` | up: occupancy |
| corr / pc1 off | `job_size` spread (beta a,b) | narrower -> higher corr |
| total_mean off | `job_power` mean or occupancy | scale to 16.1 MW |
| per_rack_mean jitter | average more seeds | -- |

`_job_profile` (intra-job shape) is **not** a cfg knob -- it's a function in `generator.py`.

In [ ]:
# Baseline: the constraint-derived starting theta
cfg = wg.WorkloadConfig.regime_A_starting(SPEC)
evaluate(cfg);

In [ ]:
# === TWEAK CELL: edit knobs, re-run, watch Q ===
cfg = wg.WorkloadConfig.regime_A_starting(SPEC)        # start fresh (comment out to keep tuning the same cfg)

cfg = dataclasses.replace(                             # replace() rebuilds -> re-runs validation
    cfg,
    noise_amp_W = 30_000.0,
    # arrival_rate_per_s = 3.0e-4,
    # job_power = wg.DistSpec("normal", {"mean": 756_000, "std": 50_000}),
    # duration  = wg.DistSpec("lognormal", {"mu": 7.62, "sigma": 1.0}),
    # job_size  = wg.DistSpec("beta", {"a": 30, "b": 1.5}),
    # placement = "contiguous",
)
evaluate(cfg);

## Tips
- **Keep seeds fixed** while turning one knob, so Q changes only from your edit (common random numbers).
- A setting is "passing" only if it passes across **multiple seeds**.
- Read the **dominant squared term** in Q to choose the next knob.
- Knobs **interact** (occupancy moves both total_mean and busy-fraction) -- adjust one at a time.

In [ ]:
# Freeze the calibrated config once seeds pass
out = Path(wg.__file__).parent / "spec" / "regime_A_config.json"
cfg.to_json(out)
print("saved", out)